# Verifier Primacy A/B Analysis

Analyzes the thesis: **a compact rule-based verifier can significantly improve agent policy compliance at near-zero cost.**

Compares two configurations:
- **baseline**: Agent without verifier
- **with-rules-verifier**: Agent with rule-based verifier intercepting tool calls

In [ ]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load results - adjust path to your most recent run
results_base = Path("../results")
experiment_dirs = sorted(
    [d for d in results_base.iterdir() if d.is_dir() and "verifier-primacy" in d.name],
    reverse=True
)

if not experiment_dirs:
    raise FileNotFoundError("No results found. Run the experiment first.")

results_dir = experiment_dirs[0]
print(f"Loading from: {results_dir}")

with open(results_dir / "summary.json") as f:
    summary = json.load(f)

summary

## 1. Task Success Rate Comparison

In [ ]:
configs = summary["configs"]
config_names = list(configs.keys())

metrics_df = pd.DataFrame({
    name: configs[name]["aggregateMetrics"]
    for name in config_names
}).T

metrics_df.index.name = "config"
metrics_df

In [ ]:
# Bar chart: key metrics comparison
key_metrics = ["task_success_rate", "policy_violation_rate", "autonomy_score"]
available_metrics = [m for m in key_metrics if m in metrics_df.columns]

fig = go.Figure()
for config in config_names:
    fig.add_trace(go.Bar(
        name=config,
        x=available_metrics,
        y=[metrics_df.loc[config, m] for m in available_metrics],
    ))

fig.update_layout(
    title="Key Metrics: Baseline vs With Verifier",
    barmode="group",
    yaxis_title="Rate",
    yaxis_tickformat=".0%",
)
fig.show()

## 2. Policy Violation Rate

In [ ]:
if "policy_violation_rate" in metrics_df.columns:
    fig = px.bar(
        x=config_names,
        y=[metrics_df.loc[c, "policy_violation_rate"] for c in config_names],
        labels={"x": "Configuration", "y": "Policy Violation Rate"},
        title="Policy Violation Rate by Configuration",
        color=config_names,
    )
    fig.update_layout(yaxis_tickformat=".0%")
    fig.show()

## 3. Verifier Intervention Analysis

In [ ]:
# Load verifier log
verifier_log_path = results_dir / "verifier-log.json"
if verifier_log_path.exists():
    with open(verifier_log_path) as f:
        verifier_log = json.load(f)

    vlog_df = pd.DataFrame(verifier_log)
    print(f"Total verifier entries: {len(vlog_df)}")

    if not vlog_df.empty:
        # Rule trigger frequency
        rule_counts = vlog_df.groupby(["ruleId", "passed"]).size().unstack(fill_value=0)
        print("\nRule trigger counts:")
        display(rule_counts)

        # Bar chart of rule failures
        failures = vlog_df[~vlog_df["passed"]]
        if not failures.empty:
            fig = px.histogram(
                failures,
                x="ruleId",
                title="Verifier Rule Failures by Rule",
                labels={"ruleId": "Rule ID", "count": "Failure Count"},
            )
            fig.show()
else:
    print("No verifier log found.")

## 4. Cost & Latency

In [ ]:
cost_metrics = ["cost_per_task", "mean_turns_to_resolution"]
available_cost = [m for m in cost_metrics if m in metrics_df.columns]

if available_cost:
    fig = make_subplots(rows=1, cols=len(available_cost),
                        subplot_titles=available_cost)

    for i, metric in enumerate(available_cost, 1):
        for config in config_names:
            fig.add_trace(
                go.Bar(name=config, x=[config], y=[metrics_df.loc[config, metric]]),
                row=1, col=i
            )

    fig.update_layout(title="Cost & Efficiency Metrics", showlegend=False)
    fig.show()

## 5. Pass@k Curves

In [ ]:
# Load individual traces to compute pass@k for different k values
traces_dir = results_dir / "traces"
if traces_dir.exists():
    traces = []
    for f in sorted(traces_dir.iterdir()):
        if f.suffix == ".json":
            with open(f) as fh:
                traces.append(json.load(fh))

    print(f"Loaded {len(traces)} traces")
else:
    print("No traces directory found.")

## 6. The Thesis: Verifier Delta

In [ ]:
delta = summary.get("verifier_delta", {})
if delta:
    print("Verifier Delta (with-verifier minus baseline):")
    print("="*50)
    for metric, value in delta.items():
        sign = "+" if value >= 0 else ""
        if abs(value) < 1:
            print(f"  {metric}: {sign}{value*100:.1f} percentage points")
        else:
            print(f"  {metric}: {sign}{value:.3f}")

    # Key thesis metric
    pvr_delta = delta.get("policy_violation_rate_delta", 0)
    tsr_delta = delta.get("task_success_rate_delta", 0)
    cost_delta = delta.get("cost_per_task_delta", 0)

    print(f"\n{'='*50}")
    print(f"THESIS VALIDATION:")
    print(f"  Policy violation reduction: {abs(pvr_delta)*100:.1f}pp")
    print(f"  Task success change: {'+' if tsr_delta >= 0 else ''}{tsr_delta*100:.1f}pp")
    print(f"  Cost change per task: ${cost_delta:.4f}")
else:
    print("No verifier delta computed (need exactly 2 configs).")